# MVGC2 reproducible tutorial — ComplexTorch parity walk-through

This notebook reproduces the structure and qualitative outputs of `complexbox/examples/mvgc_tutorial.ipynb` using the Torch-first `complextorch` API.

1. Generate a stationary random VAR(p).
2. Simulate multi-trial data.
3. Select model order with AIC/BIC/HQC.
4. Fit LWR and OLS models.
5. Diagnose consistency and whiteness.
6. Compute time-domain pairwise-conditional GC.
7. Compute spectral GC and integrate it.
8. Test significance with FDR.
9. Verify VAR and innovations-state-space equivalence.

ComplexTorch uses `(trials, time, variables)` and `(batch, lag, target, source)`. The displayed GC orientation remains `F[i,j] = j -> i`.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import complextorch as ct
from complextorch.measures import integrate_spectral_mvgc
torch.set_default_dtype(torch.float64)
np.set_printoptions(precision=4,suppress=True)
dtype=torch.float64
device="cpu"


## 1. Random ground-truth VAR model

A five-variable VAR(3) is scaled to spectral radius 0.9. The innovation covariance is sampled with the onion method.


In [ ]:
n_vars,p_true=5,3
A_true,_=ct.random_stable_var(batch=1,n_variables=n_vars,order=p_true,spectral_radius_target=.9,seed=20260516,dtype=dtype,device=device)
V_true=ct.random_correlation_matrix(n_variables=n_vars,batch=1,seed=20260517,dtype=dtype,device=device)
true_system=ct.build_var_system(A_true,V_true)
print(f"spectral radius = {float(true_system.spectral_radius[0]):.6f}")
print("residuals covariance V_true:")
print(V_true[0].numpy())


## 2. Simulate data

The simulator uses MVGC-style automatic transient truncation and optionally returns the innovations.


In [ ]:
m=10_000
N=1
X,E=ct.simulate_var(A_true,V_true,n_times=m,burnin="auto",seed=20260518,return_innovations=True)
print("X shape:",tuple(X.shape))
fig,axes=plt.subplots(n_vars,1,figsize=(10,6),sharex=True)
for i,ax in enumerate(axes):
    ax.plot(X[0,:500,i].numpy(),lw=.7); ax.set_ylabel(f"x{i+1}")
axes[-1].set_xlabel("time sample")
fig.suptitle("first 500 samples of simulated VAR data")
fig.tight_layout()


## 3. Model-order selection

`VAROrderSelectionIC` implements the MVGC2 AIC, BIC and HQC definitions.


In [ ]:
mo=ct.VAROrderSelectionIC(orders=range(1,11),solver="lwr",refit="hqc",device=device,dtype="float64").fit(X)
print(f"AIC -> p = {mo.p_aic_}")
print(f"BIC -> p = {mo.p_bic_}")
print(f"HQC -> p = {mo.p_hqc_}")
print(f"truth: p = {p_true}")
fig,ax=plt.subplots(figsize=(8,4))
p_axis=np.asarray(mo.result_.orders)
ax.plot(p_axis,mo.aic_,"o-",label="AIC")
ax.plot(p_axis,mo.bic_,"s-",label="BIC")
ax.plot(p_axis,mo.hqc_,"^-",label="HQC")
ax.axvline(p_true,ls="--",c="k",alpha=.5,label=f"true order p={p_true}")
ax.set_xlabel("VAR order p"); ax.set_ylabel("information criterion (per-observation)")
ax.legend(); ax.set_title("VAR model-order selection")


## 4. Fit the VAR using LWR and OLS

`solver="lwr"` implements Morf lattice-whitening regression. `solver="lstsq"` performs ordinary least squares.


In [ ]:
p=mo.p_hqc_
fit_lwr=ct.VAR(order=p,solver="lwr",covariance="unbiased",fit_intercept=True,mode="pooled",device=device,dtype="float64").fit(X)
fit_ols=ct.VAR(order=p,solver="lstsq",covariance="unbiased",fit_intercept=True,mode="pooled",device=device,dtype="float64").fit(X)
print(f"fit p = {p}")
print(f"rho(A_LWR) = {float(fit_lwr.spectral_radius_[0]):.6f}")
print(f"rho(A_OLS) = {float(fit_ols.spectral_radius_[0]):.6f}")
if p>=p_true:
    err=torch.max(torch.abs(fit_lwr.coef_[0,:p_true]-A_true[0]))
    print(f"max |A_LWR - A_true| over first {p_true} lags = {float(err):.4e}")


## 5. Diagnostics: consistency and whiteness

The Ding–Bressler consistency statistic should generally exceed 0.8 for a good fit. Durbin–Watson is reported per variable.


In [ ]:
cons=fit_lwr.consistency(X)
white=fit_lwr.whiteness(X,method="durbin_watson")
print(f"consistency = {cons:.4f}")
print("Durbin-Watson statistics per variable:",white.statistic.numpy())
print("whiteness p-values:                  ",white.pvalue.numpy())


## 6. Time-domain pairwise-conditional Granger causality

Each ordered pair conditions on all remaining variables. The diagonal is undefined.


In [ ]:
def pwcgc(model,n):
    F=torch.full((n,n),torch.nan,dtype=dtype)
    for target in range(n):
        for source in range(n):
            if target==source: continue
            conditional=tuple(k for k in range(n) if k not in (target,source))
            F[target,source]=ct.temporal_mvgc(model,source=(source,),target=(target,),conditional=conditional).reshape(-1)[0]
    return F
F_true=pwcgc(true_system,n_vars)
F_hat=pwcgc(fit_lwr.to_var_system(),n_vars)
fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,mat,title in zip(axes,(F_true,F_hat),("ground truth","LWR estimate")):
    im=ax.imshow(torch.nan_to_num(mat,nan=0.).numpy(),cmap="viridis",vmin=0)
    plt.colorbar(im,ax=ax); ax.set_title(f"pairwise-conditional GC — {title}")
    ax.set_xlabel("source j"); ax.set_ylabel("target i")
fig.tight_layout()


## 7. Spectral Granger causality

ComplexTorch uses normalized frequencies `f in [0,.5]`; the plot displays `omega=2*pi*f` for MVGC2 parity.


In [ ]:
fres=256
frequencies=torch.linspace(0.,.5,fres+1,dtype=dtype)
omega=2*torch.pi*frequencies
F_spec=torch.full((n_vars,n_vars,fres+1),torch.nan,dtype=dtype)
for target in range(n_vars):
    for source in range(n_vars):
        if target==source: continue
        conditional=tuple(k for k in range(n_vars) if k not in (target,source))
        value=ct.spectral_mvgc(fit_lwr.to_var_system(),source=(source,),target=(target,),conditional=conditional,frequencies=frequencies)
        F_spec[target,source]=value.reshape(-1,fres+1)[0]
fig,ax=plt.subplots(figsize=(10,4))
for target in range(n_vars):
    for source in range(n_vars):
        if target!=source: ax.plot(omega.numpy(),F_spec[target,source].numpy(),lw=.8,label=f"{source+1} -> {target+1}")
ax.set_xlabel("frequency (rad)"); ax.set_ylabel("spectral GC")
ax.set_title("pairwise-conditional spectral GC"); ax.legend(ncol=2,fontsize=8)
F_int=torch.full_like(F_hat,torch.nan)
for target in range(n_vars):
    for source in range(n_vars):
        if target!=source: F_int[target,source]=integrate_spectral_mvgc(F_spec[target,source],frequencies)
diff=torch.nan_to_num(F_int-F_hat,nan=0.).abs().max()
print("max |integrated spectral GC - time-domain GC| =",float(diff))


## 8. Statistical significance testing

P-values use the MVGC2 F-test convention and Benjamini–Hochberg FDR correction.


In [ ]:
pvals=np.full((n_vars,n_vars),np.nan)
for target in range(n_vars):
    for source in range(n_vars):
        if target==source: continue
        pvals[target,source]=ct.mvgc_pvalue(float(F_hat[target,source]),method="F",n_target=1,n_source=1,n_conditional=n_vars-2,order=p,n_times=m,n_trials=N)
sig=ct.significance(pvals,alpha=.05,method="fdr_bh")
print("FDR-significant connections (i <- j):")
print(sig.astype(int))


## 9. State-space pathway — same GC, different route

The fitted VAR is converted exactly to innovations-form state space. GC must agree to machine precision.


In [ ]:
iss=ct.var_to_innovations_state_space(fit_lwr.to_var_system())
F_ss=torch.full_like(F_hat,torch.nan)
for target in range(n_vars):
    for source in range(n_vars):
        if target==source: continue
        conditional=tuple(k for k in range(n_vars) if k not in (target,source))
        F_ss[target,source]=ct.temporal_mvgc(iss,source=(source,),target=(target,),conditional=conditional).reshape(-1)[0]
diff=torch.nan_to_num(F_hat-F_ss,nan=0.).abs().max()
print(f"max |F_VAR - F_SS| = {float(diff):.4e}  (should be machine precision)")
assert float(diff)<1e-8


## Validation boundary

Exact numerical equality with an independent ComplexBox run requires shared fixtures for the true VAR, covariance and simulated innovations. The invariant checks here are spectral/time GC agreement, VAR/ISS agreement, stable LWR estimation and identical MVGC2 statistical conventions.

Further reading: Barnett & Seth (2014), *J. Neurosci. Methods*; Barnett & Seth (2015), *Phys. Rev. E*.
